In [1]:
from random import seed

import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/processed/ealaxi/paysim1/data.csv")
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,errorbalanceOrg,errorbalanceDest,HourOfDay
0,1,TRANSFER,181.00,181.0,0.0,0.0,0.00,1,0.00,181.0,1
1,1,CASH_OUT,181.00,181.0,0.0,21182.0,0.00,1,0.00,21363.0,1
2,1,CASH_OUT,229133.94,15325.0,0.0,5083.0,51513.44,0,213808.94,182703.5,1
3,1,TRANSFER,215310.30,705.0,0.0,22425.0,0.00,0,214605.30,237735.3,1
4,1,TRANSFER,311685.89,10835.0,0.0,6267.0,2719172.89,0,300850.89,-2401220.0,1


In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["type"] = le.fit_transform(df["type"])

In [4]:
X = df.drop("isFraud", axis=1)
Y = df.isFraud

seed(21)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [5]:
print("Shape of X_train: ", X_train.shape)
print("Shape of X_test: ", X_test.shape)

Shape of X_train:  (2216327, 10)
Shape of X_test:  (554082, 10)


In [6]:
from sklearn.ensemble import RandomForestClassifier  # Random forest tree algorithm
from sklearn.metrics import (
    auc,
    classification_report,
    confusion_matrix,
    roc_curve,
)
from sklearn.tree import DecisionTreeClassifier  # Decision tree algorithm
from xgboost import XGBClassifier  # XGBoost algorithm

In [7]:
# Random Forest
parameters_rf = {
    'n_estimators': 15,
    'class_weight': 'balanced',
    'n_jobs': -1,
    'random_state': 42
}
RF = RandomForestClassifier(**parameters_rf)
fitted_values = RF.fit(X_train, y_train)

predictions_rf = RF.predict(X_test)

confusion_matrx_rf = confusion_matrix(y_test, predictions_rf)
classification_report_rf = classification_report(y_test, predictions_rf)
fpr_rf, recall_rf, thresholds_rf = roc_curve(y_test, predictions_rf)
auc_rf = auc(fpr_rf, recall_rf)

results_rf = {"Confusion Matrix":confusion_matrx_rf,"Classification Report":classification_report_rf,"Area Under Curve":auc_rf}

for measure in results_rf:
    print(measure,": \n",results_rf[measure])


Confusion Matrix : 
 [[552435      1]
 [    10   1636]]
Classification Report : 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    552436
           1       1.00      0.99      1.00      1646

    accuracy                           1.00    554082
   macro avg       1.00      1.00      1.00    554082
weighted avg       1.00      1.00      1.00    554082

Area Under Curve : 
 0.9969614278460933


In [9]:
# Train model
DT = DecisionTreeClassifier(class_weight='balanced')
fitted_vals = DT.fit(X_train, y_train)

# Predict on testing set
predictionsDT = DT.predict(X_test)


# Evaluating model
CM_DT = confusion_matrix(y_test,predictionsDT)
CR_DT = classification_report(y_test,predictionsDT)
fprDT, recallDT, thresholdsDT = roc_curve(y_test, predictionsDT)
AUC_DT = auc(fprDT, recallDT)

resultsDT = {"Confusion Matrix":CM_DT,"Classification Report":CR_DT,"Area Under Curve":AUC_DT}

# showing results from Random Forest

for measure in resultsDT:
    print(measure,": \n",resultsDT[measure])


Confusion Matrix : 
 [[552427      9]
 [    12   1634]]
Classification Report : 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    552436
           1       0.99      0.99      0.99      1646

    accuracy                           1.00    554082
   macro avg       1.00      1.00      1.00    554082
weighted avg       1.00      1.00      1.00    554082

Area Under Curve : 
 0.9963466537740142


In [14]:
# Train model
parametersXGB = {'max_depth':100,'class_weight': "balanced",'n_jobs':-1,'random_state':42,'learning_rate':0.1}
XGB = XGBClassifier(**parametersXGB)


fitted_vals = XGB.fit(X_train, y_train)

# Predict on testing set
predictionsXGB = XGB.predict(X_test)


# Evaluating model
CM_XGB = confusion_matrix(y_test,predictionsXGB)
CR_XGB = classification_report(y_test,predictionsXGB)
fprXGB, recallXGB, thresholds_XGB = roc_curve(y_test, predictionsXGB)
AUC_XGB = auc(fprXGB, recallXGB)
resultsXGB = {"Confusion Matrix":CM_XGB,"Classification Report":CR_XGB,"Area Under Curve":AUC_XGB}
# showing results from Extreme Gradient Boosting
for measure in resultsXGB:
    print(measure,": \n",resultsXGB[measure],"\n")


/home/shan-qing/miniconda3/envs/prac_mlops/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [19:20:55] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1772125072520/work/src/learner.cc:782: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Confusion Matrix : 
 [[552383     53]
 [    55   1591]] 

Classification Report : 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    552436
           1       0.97      0.97      0.97      1646

    accuracy                           1.00    554082
   macro avg       0.98      0.98      0.98    554082
weighted avg       1.00      1.00      1.00    554082
 

Area Under Curve : 
 0.9832448617481744 



In [16]:
from collections import Counter

from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_resampled, Y_resampled = rus.fit_resample(X, Y)
print("Resampled shape of X: ", X_resampled.shape)
print("Resampled shape of Y: ", Y_resampled.shape)
value_counts = Counter(Y_resampled)
print(value_counts)
train_X, test_X, train_Y, test_Y = train_test_split(X_resampled, Y_resampled, test_size= 0.3, random_state= 42)


Resampled shape of X:  (16426, 10)
Resampled shape of Y:  (16426,)
Counter({0: 8213, 1: 8213})


In [17]:
# Train model
parametersRF = {'n_estimators':15,'n_jobs':-1,'random_state':42}
RF = RandomForestClassifier(**parametersRF)
fitted_vals = RF.fit(train_X, train_Y)

# Predict on testing set
predictionsRF = RF.predict(test_X)


# Evaluating model
CM_RF = confusion_matrix(test_Y,predictionsRF)
CR_RF = classification_report(test_Y,predictionsRF)
fprRF, recallRF, thresholdsRF = roc_curve(test_Y, predictionsRF)
AUC_RF = auc(fprRF, recallRF)

resultsRF = {"Confusion Matrix":CM_RF,"Classification Report":CR_RF,"Area Under Curve":AUC_RF}

# showing results from Random Forest

for measure in resultsRF:
    print(measure,": \n",resultsRF[measure])

Confusion Matrix : 
 [[2474    5]
 [   8 2441]]
Classification Report : 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      2479
           1       1.00      1.00      1.00      2449

    accuracy                           1.00      4928
   macro avg       1.00      1.00      1.00      4928
weighted avg       1.00      1.00      1.00      4928

Area Under Curve : 
 0.9973582091199394


In [18]:
# Train model
DT = DecisionTreeClassifier()
fitted_vals = DT.fit(X_train, y_train)

# Predict on testing set
predictionsDT = DT.predict(X_test)


# Evaluating model
CM_DT = confusion_matrix(y_test,predictionsDT)
CR_DT = classification_report(y_test,predictionsDT)
fprDT, recallDT, thresholdsDT = roc_curve(y_test, predictionsDT)
AUC_DT = auc(fprDT, recallDT)

resultsDT = {"Confusion Matrix":CM_DT,"Classification Report":CR_DT,"Area Under Curve":AUC_DT}

# showing results from Random Forest

for measure in resultsDT:
    print(measure,": \n",resultsDT[measure])

Confusion Matrix : 
 [[552426     10]
 [    11   1635]]
Classification Report : 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    552436
           1       0.99      0.99      0.99      1646

    accuracy                           1.00    554082
   macro avg       1.00      1.00      1.00    554082
weighted avg       1.00      1.00      1.00    554082

Area Under Curve : 
 0.9966495153989655


In [21]:
# Train model
parametersXGB = {'max_depth':10,'n_jobs':-1,'random_state':42,'learning_rate':0.1, 'device':'cuda'}
XGB = XGBClassifier(**parametersXGB)


fitted_vals = XGB.fit(X_train, y_train)

# Predict on testing set
predictionsXGB = XGB.predict(X_test)


# Evaluating model
CM_XGB = confusion_matrix(y_test,predictionsXGB)
CR_XGB = classification_report(y_test,predictionsXGB)
fprXGB, recallXGB, thresholds_XGB = roc_curve(y_test, predictionsXGB)
AUC_XGB = auc(fprXGB, recallXGB)
resultsXGB = {"Confusion Matrix":CM_XGB,"Classification Report":CR_XGB,"Area Under Curve":AUC_XGB}
# showing results from Extreme Gradient Boosting
for measure in resultsXGB:
    print(measure,": \n",resultsXGB[measure],"\n")

Confusion Matrix : 
 [[552383     53]
 [    53   1593]] 

Classification Report : 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00    552436
           1       0.97      0.97      0.97      1646

    accuracy                           1.00    554082
   macro avg       0.98      0.98      0.98    554082
weighted avg       1.00      1.00      1.00    554082
 

Area Under Curve : 
 0.9838523951625122 



/home/shan-qing/miniconda3/envs/prac_mlops/lib/python3.10/site-packages/xgboost/core.py:751: UserWarning: [19:57:25] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1772125072520/work/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
